# Лабораторная работа №3. Арифметические вычисления

**Выполнил:** Рулев Константин Александрович  
**Вариант:** 16

**Цель:** вывести ASCII-код символа, вычислить выражение на ассемблере и сравнить результат с программой на C.

Отчёт содержит исходники и команды запуска. В Google Colab выберите обычную CPU-среду и выполните все ячейки по порядку.


Результаты получены при выполнении в Linux x86-64. Все файлы для повторного запуска встроены в отчёт.


## 1. Задания и вариант

1. Передать символ через командную строку и вывести его ASCII-код.
2. Вычислить выражение `((((b/b)/a)/a)/c)` на ассемблере. Числа `a`, `b`, `c` передать через командную строку.
3. Написать такую же программу на C и сравнить результаты.


In [1]:
import random as rd

Variant = 16
rd.seed(Variant)
set_operations = ['-', '+', '*', '/']
set_operands = ['a', 'b', 'c']
count_operations = rd.randint(3, 5)
expression = set_operands[rd.randint(0, len(set_operands) - 1)]
for i in range(count_operations):
    operation = set_operations[rd.randint(0, len(set_operations) - 1)]
    operand = set_operands[rd.randint(0, len(set_operands) - 1)]
    expression = '(' + expression + operation + operand + ')'
print(expression)


((((b/b)/a)/a)/c)


## 2. Алгоритм

В первом задании программа берёт один символ из аргумента и печатает его код. Допустим один ASCII-символ с кодом от 1 до 127. Код 0 нельзя передать в аргументе командной строки.

Во втором задании строки преобразуются в целые числа. Деление выполняется по порядку скобок, дробная часть отбрасывается к нулю. Ввод: целые числа от −1 000 000 000 до 1 000 000 000. При нулевом делителе или неверном вводе программа завершится с кодом 1.


## 3. Подготовка среды


In [2]:
import os
import platform
import shutil
import subprocess
from pathlib import Path

if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'amd64'):
    raise RuntimeError('Нужна Linux x86-64 CPU-среда Google Colab')
if not all(shutil.which(name) for name in ('fasm', 'gcc', 'ld', 'make')):
    prefix = [] if os.geteuid() == 0 else ['sudo']
    subprocess.run(prefix + ['apt-get', 'update', '-qq'], check=True)
    subprocess.run(prefix + ['apt-get', 'install', '-y', '-qq', 'fasm', 'gcc', 'binutils', 'make'], check=True)
base = Path('/content') if Path('/content').exists() else Path('/tmp')
work = base / 'rulev_variant16' / 'Lab_3'
work.mkdir(parents=True, exist_ok=True)
os.chdir(work)
print('Среда готова')


Среда готова


## 4. Исходные тексты

`io.inc` содержит преобразование чисел, вывод и завершение программы.


### io.inc


In [3]:
%%writefile io.inc
section '.data' writable
error_message db 'Error: invalid input', 10
error_length = $ - error_message

section '.bss' writable
number_buffer rb 32

section '.text' executable
parse_number:
    xor rax, rax
    mov r8, 1
    cmp byte [rsi], '-'
    jne .plus
    mov r8, -1
    inc rsi
    jmp .first
.plus:
    cmp byte [rsi], '+'
    jne .first
    inc rsi
.first:
    cmp byte [rsi], 0
    je error
.digit:
    xor r9, r9
    mov r9b, [rsi]
    cmp r9b, '0'
    jb error
    cmp r9b, '9'
    ja error
    sub r9, '0'
    imul rax, 10
    add rax, r9
    cmp rax, 1000000000
    ja error
    inc rsi
    cmp byte [rsi], 0
    jne .digit
    imul rax, r8
    ret

print_number:
    mov rsi, number_buffer + 31
    mov byte [rsi], 10
    mov r8, rax
    cmp rax, 0
    jge .digits
    neg rax
.digits:
    mov r9, 10
.next:
    xor rdx, rdx
    div r9
    add dl, '0'
    dec rsi
    mov [rsi], dl
    cmp rax, 0
    jne .next
    cmp r8, 0
    jge .write
    dec rsi
    mov byte [rsi], '-'
.write:
    mov rdx, number_buffer + 32
    sub rdx, rsi
    mov rax, 1
    mov rdi, 1
    syscall
    ret

exit:
    mov rax, 60
    xor rdi, rdi
    syscall

error:
    mov rax, 1
    mov rdi, 2
    mov rsi, error_message
    mov rdx, error_length
    syscall
    mov rax, 60
    mov rdi, 1
    syscall


Writing io.inc


### task1.asm


In [4]:
%%writefile task1.asm
format ELF64
public _start

section '.text' executable
_start:
    cmp qword [rsp], 2
    jne error
    mov rsi, [rsp + 16]
    cmp byte [rsi], 0
    je error
    cmp byte [rsi + 1], 0
    jne error
    xor rax, rax
    mov al, [rsi]
    cmp rax, 127
    ja error
    call print_number
    call exit

include 'io.inc'


Writing task1.asm


### task2.asm


In [5]:
%%writefile task2.asm
format ELF64
public _start

section '.text' executable
_start:
    cmp qword [rsp], 4
    jne error
    mov rsi, [rsp + 16]
    call parse_number
    mov r12, rax
    mov rsi, [rsp + 24]
    call parse_number
    mov r13, rax
    mov rsi, [rsp + 32]
    call parse_number
    mov r14, rax
    cmp r12, 0
    je error
    cmp r13, 0
    je error
    cmp r14, 0
    je error
    mov rax, r13
    mov rdx, rax
    sar rdx, 63
    idiv r13
    mov rdx, rax
    sar rdx, 63
    idiv r12
    mov rdx, rax
    sar rdx, 63
    idiv r12
    mov rdx, rax
    sar rdx, 63
    idiv r14
    call print_number
    call exit

include 'io.inc'


Writing task2.asm


### task2.c


In [6]:
%%writefile task2.c
#include <stdio.h>

int parse_number(const char *s, long long *result) {
    long long value = 0;
    int sign = 1;
    if (*s == '-' || *s == '+') {
        if (*s == '-') sign = -1;
        ++s;
    }
    if (*s == '\0') return 0;
    while (*s) {
        if (*s < '0' || *s > '9') return 0;
        value = value * 10 + *s - '0';
        if (value > 1000000000) return 0;
        ++s;
    }
    *result = value * sign;
    return 1;
}

int main(int argc, char **argv) {
    long long a, b, c;
    if (argc != 4 || !parse_number(argv[1], &a) ||
        !parse_number(argv[2], &b) || !parse_number(argv[3], &c) ||
        a == 0 || b == 0 || c == 0) {
        fputs("Error: invalid input\n", stderr);
        return 1;
    }
    printf("%lld\n", (((b / b) / a) / a) / c);
    return 0;
}


Writing task2.c


### Makefile


In [7]:
%%writefile Makefile
.PHONY: all check clean

all: build/task1 build/task2 build/task2_c

build:
	mkdir -p build

build/%.o: %.asm io.inc | build
	fasm $< $@

build/task1: build/task1.o
	ld -z noexecstack -o $@ $<

build/task2: build/task2.o
	ld -z noexecstack -o $@ $<

build/task2_c: task2.c | build
	gcc -std=c11 -Wall -Wextra -Werror -O0 $< -o $@

check: all
	python3 check.py

clean:
	rm -rf build


Writing Makefile


### check.py


In [8]:
%%writefile check.py
from pathlib import Path
import random
import subprocess

root = Path(__file__).resolve().parent
count = 0


def run(name, args, expected=None):
    global count
    result = subprocess.run(
        [str(root / 'build' / name), *map(str, args)],
        capture_output=True, text=True, timeout=3
    )
    if expected is None:
        assert result.returncode == 1, (name, args, result)
        assert result.stdout == '' and result.stderr == 'Error: invalid input\n'
    else:
        assert result.returncode == 0, (name, args, result)
        assert result.stdout == f'{expected}\n', (name, args, result.stdout, expected)
        assert result.stderr == ''
    count += 1


def divide(a, b):
    value = abs(a) // abs(b)
    return -value if (a < 0) != (b < 0) else value


for value in range(1, 128):
    run('task1', [chr(value)], value)
for args in ([], [''], ['AB'], ['А'], ['é'], ['A', 'B']):
    run('task1', args)

values = [
    (1, 7, 1), (-1, 7, 1), (1, -7, -1), (-1, -7, -1),
    (2, 7, 1), (1, 7, 2), (20, 3, 4),
    (1000000000, -1000000000, 1000000000),
    (-1000000000, 1000000000, -1000000000),
]
rng = random.Random(16)
for _ in range(100):
    values.append(tuple(rng.choice([*range(-15, 0), *range(1, 16)]) for _ in range(3)))

for a, b, c in values:
    expected = divide(divide(divide(divide(b, b), a), a), c)
    for name in ('task2', 'task2_c'):
        run(name, [a, b, c], expected)

invalid = [
    [], [1, 2], [1, 2, 3, 4], [0, 2, 3], [1, 0, 3], [1, 2, 0],
    ['', 2, 3], ['-', 2, 3], ['+', 2, 3], ['1x', 2, 3], [' 1', 2, 3],
    ['1.5', 2, 3], [1000000001, 2, 3], [-1000000001, 2, 3],
    [1, '99999999999999999999999999999', 1], [1, 2, '--1'],
]
for name in ('task2', 'task2_c'):
    run(name, ['+1', '-0007', '-1'], -1)
    for args in invalid:
        run(name, args)

print(f'Проверки пройдены: {count}')


Writing check.py


## 5. Сборка


In [9]:
%%bash
set -e
make all


mkdir -p build
fasm task1.asm build/task1.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
5 passes, 1408 bytes.
ld -z noexecstack -o build/task1 build/task1.o
fasm task2.asm build/task2.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
5 passes, 1552 bytes.
ld -z noexecstack -o build/task2 build/task2.o
gcc -std=c11 -Wall -Wextra -Werror -O0 task2.c -o build/task2_c


## 6. Результаты

ASCII-коды букв `A`, `z` и пробела:


In [10]:
%%bash
set -e
./build/task1 A
./build/task1 z
./build/task1 ' '


65
122
32


Сравнение ASM и C. Для `(a, b, c) = (1, 7, 1)` результат равен `1`; для `(1, -7, -1)` — `-1`; для `(20, 3, 4)` — `0`.


In [11]:
import subprocess

for a, b, c in [(1, 7, 1), (1, -7, -1), (20, 3, 4)]:
    print(f'a={a}, b={b}, c={c}')
    for name in ['task2', 'task2_c']:
        result = subprocess.run([f'./build/{name}', str(a), str(b), str(c)], capture_output=True, text=True, check=True)
        print(f'{name}: {result.stdout.strip()}')


a=1, b=7, c=1
task2: 1
task2_c: 1
a=1, b=-7, c=-1
task2: -1
task2_c: -1
a=20, b=3, c=4
task2: 0
task2_c: 0


Проверка деления на ноль:


In [12]:
for name in ['task2', 'task2_c']:
    result = subprocess.run([f'./build/{name}', '1', '0', '1'], capture_output=True, text=True)
    assert result.returncode == 1
    print(f'{name}: {result.stderr.strip()}, код завершения {result.returncode}')


task2: Error: invalid input, код завершения 1
task2_c: Error: invalid input, код завершения 1


Автоматическая проверка: все допустимые ASCII-коды, положительные и отрицательные числа, границы диапазона и ошибки ввода.


In [13]:
%%bash
set -e
make check


python3 check.py
Проверки пройдены: 385


## 7. Вывод

Программа выводит ASCII-код переданного символа. Выражение вычисляется на ASM и C одинаково. Неверный ввод и деление на ноль обрабатываются сообщением об ошибке.
